In [14]:
import torch

/Users/ross/dev_workspace/tahoe/.venv/lib/python3.13/site-packages/torch/_subclasses/functional_tensor.py:279: UserWarning: Failed to initialize NumPy: No module named 'numpy' (Triggered internally at /Users/runner/work/pytorch/pytorch/pytorch/torch/csrc/utils/tensor_numpy.cpp:84.)
  cpu = _conversion_method_template(device=torch.device("cpu"))


In [ ]:
# read all text
with open("data/tinyshakespeare.txt", 'r', encoding='utf-8') as f:
    text = f.read()

print(len(text))

In [8]:
chars = sorted(list(set[str](text)))
vocab_size = len(chars)
print("all tokens: ", "".join(chars))
print("vocab size: ", vocab_size)

all tokens:  
 !$&',-.3:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz
vocab size:  65


In [13]:
char_to_id = {c: i for i, c in enumerate(chars)}
id_to_char = {i: c for i, c in enumerate(chars)}

def encode(string: str) -> list[int]:
    return [char_to_id[c] for c in string]

def decode(tokens: list[int]) -> str:
    return "".join([id_to_char[t] for t in tokens]) 

assert (encode("ab") == [char_to_id["a"], char_to_id["b"]])
assert (decode([char_to_id["a"], char_to_id["b"]]) == "ab")

In [22]:
data = torch.tensor(encode(text), dtype=torch.long)
print(data.shape, data.dtype)
print(data[:100])
assert (data.max().item() == vocab_size - 1 and data.min().item() == 0)

torch.Size([1115394]) torch.int64
tensor([18, 47, 56, 57, 58,  1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 14, 43, 44,
        53, 56, 43,  1, 61, 43,  1, 54, 56, 53, 41, 43, 43, 42,  1, 39, 52, 63,
         1, 44, 59, 56, 58, 46, 43, 56,  6,  1, 46, 43, 39, 56,  1, 51, 43,  1,
        57, 54, 43, 39, 49,  8,  0,  0, 13, 50, 50, 10,  0, 31, 54, 43, 39, 49,
         6,  1, 57, 54, 43, 39, 49,  8,  0,  0, 18, 47, 56, 57, 58,  1, 15, 47,
        58, 47, 64, 43, 52, 10,  0, 37, 53, 59])


In [23]:
# distribution
torch.bincount(data)

tensor([ 40000, 169892,   2172,      1,      3,   6187,  19846,   1897,   7885,
            27,  10316,   3628,   2462,   7819,   2761,   3820,   2089,   6041,
          1797,   2399,   3068,  11832,    320,   1584,   3876,   2840,   5079,
          5481,   1641,    231,   4869,   4523,   7015,   3313,    798,   3530,
           112,   1718,    198,  55507,  11321,  15623,  31358,  94611,  15770,
         13356,  51310,  45537,    628,   7088,  33339,  22243,  48529,  65798,
         10808,    609,  48889,  49696,  67009,  26584,   7793,  17585,    529,
         20448,    356])

In [24]:
n = int(0.9 * len(data))
train_data = data[:n]
val_data = data[n:]

In [25]:
# equivalent to context window for Transformers
block_size = 8

In [27]:
# given tokens [10, 11, 12], model(10) -> 11; ie. it's shifted one to the right
# takeaway: we care about `context_window + 1` slices.
x, y = train_data[: block_size], train_data[1: block_size + 1]
for t in range(block_size):
    context = x[:t+1]
    target = y[t]
    print(f"when input is {context} the target: {target}")

when input is tensor([18]) the target: 47
when input is tensor([18, 47]) the target: 56
when input is tensor([18, 47, 56]) the target: 57
when input is tensor([18, 47, 56, 57]) the target: 58
when input is tensor([18, 47, 56, 57, 58]) the target: 1
when input is tensor([18, 47, 56, 57, 58,  1]) the target: 15
when input is tensor([18, 47, 56, 57, 58,  1, 15]) the target: 47
when input is tensor([18, 47, 56, 57, 58,  1, 15, 47]) the target: 58


In [ ]:
torch.manual_seed(1337)
batch_size = 4 # number of sequences to process at once
block_size = 8 # max context length for predictions

def get_batch(split: str):
    data = train_data if split == "train" else val_data
    idx = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i: i + block_size] for i in idx])
    y = torch.stack([data[i+1: i + block_size + 1] for i in idx])
    return x, y

xb, yb = get_batch("train")
print('inputs: ',  xb.shape, xb)
print('targets: ',  yb.shape, yb)

inputs:  torch.Size([4, 8]) tensor([[24, 43, 58,  5, 57,  1, 46, 43],
        [44, 53, 56,  1, 58, 46, 39, 58],
        [52, 58,  1, 58, 46, 39, 58,  1],
        [25, 17, 27, 10,  0, 21,  1, 54]])
targets:  torch.Size([4, 8]) tensor([[43, 58,  5, 57,  1, 46, 43, 39],
        [53, 56,  1, 58, 46, 39, 58,  1],
        [58,  1, 58, 46, 39, 58,  1, 46],
        [17, 27, 10,  0, 21,  1, 54, 39]])


In [31]:
import torch.nn as nn
from torch.nn import functional as F
torch.manual_seed(1337)

In [37]:
class BigramLanguageModel(nn.Module):
    def __init__(self, vocab_size: int):
        super().__init__()
        # lookup table for bigram probabilities
        self.token_table = nn.Embedding(vocab_size, vocab_size)

    def forward(self, X: torch.tensor):
        # (batch_size, block_size, vocab_size)
        return self.token_table(X)

    def generate(self, X: torch.tensor, max_new_tokens: int):
        # X: (batch_size, T)
        for _ in range(max_new_tokens):
            logits = self(X) # (batch_size, T, vocab_size)
            next_token_lh = F.softmax(logits[:, -1, :], dim=-1) # (batch_size, vocab_size)
            X_next = torch.multinomial(next_token_lh, num_samples=1)
            X = torch.cat((X, X_next), dim=1) # (batch_size, T+1)
        return X

bigram_model = BigramLanguageModel(vocab_size)
logits = bigram_model(xb)

# exercise: think about what an upper bound on the loss should be (hint. consider the uniform distribution)
B, T, C = logits.shape
logits = logits.view(B*T, C)
targets =yb.view(B*T)  
loss = F.cross_entropy(logits, targets)

print(loss)

print(
    decode(bigram_model.generate(torch.ones((1, 1), dtype=torch.long), max_new_tokens=100)[0].tolist())
)

tensor(4.9095, grad_fn=<NllLossBackward0>)
 NZqlz':Pk.RPmfk3z3yDvH-&&XbIzNeblzfavPPAv'wvHx!P&
Tj
dpnFH,PgcCqiqOPVGWNLwmBwOzg-Mjj
c!Cher.lnFf!gAp


In [39]:
optimizer = torch.optim.AdamW(bigram_model.parameters(), lr=1e-3)

In [44]:
batch_size = 32
for steps in range(10000):
    xb, yb = get_batch("train")
    logits = bigram_model(xb)

    B, T, C = logits.shape
    logits = logits.view(B*T, C)
    targets = yb.view(B*T)  
    loss = F.cross_entropy(logits, targets)

    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()
print(loss.item())

2.479444742202759


In [46]:
print(
    decode(bigram_model.generate(torch.ones((1, 1), dtype=torch.long), max_new_tokens=300)[0].tolist())
)

 briongharr my tomoued aitharary tods theacan Citorat wo t WAUGBrfack; he henous Weownk e EOKEMuleaibes:
KIfl.
Bute H:
Way pto pofathom mis wed,
WAn blelag is youes I:
Y:
LYCar?
WAheea
IORUStawof der n?
ANotibouthar,
's yowe ath glcoos as ke mer feeru test woues fe ce h san ss she wey thst thit galma


## Attention

In [ ]:
xbow = torch.zeros((4, 8, 2))
for b in range(4):
    for t in range(8):
        xprev = x